# 01 Data Acquisition

Notebook-first walkthrough for branch `02-data-ingestion`. The goal is to inspect the structured ingestion flow before relying on the script entry point.

## 1. Configure the Run

Use fixture mode for deterministic local development. Switch to live mode when you want fresh yfinance and SEC EDGAR data.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from constants import INGESTION_MODE_FIXTURE  # noqa: E402
from ingestion.pipeline import DEFAULT_FIXTURE_PATH, run_ingestion  # noqa: E402

MODE = INGESTION_MODE_FIXTURE
DB_PATH = PROJECT_ROOT / "data" / "processed" / "notebook_ingestion.sqlite"
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
TICKERS = ["MSFT", "NVDA"]
START_DATE = "2024-01-02"
END_DATE = "2024-01-05"
REFRESH_UNIVERSE = False

## 2. Inspect the Fixture Payload

Fixture mode uses committed test data, but it still goes through the same normalization and SQLite persistence code as live mode.

In [ ]:
import json

fixture_payload = json.loads(DEFAULT_FIXTURE_PATH.read_text(encoding="utf-8"))
fixture_payload.keys()

In [ ]:
fixture_payload["companies"]

## 3. Run the Shared Ingestion Pipeline

The notebook intentionally calls `run_ingestion()` rather than duplicating the implementation. This keeps the notebook, tests, and future script behavior aligned.

In [ ]:
result = run_ingestion(
    mode=MODE,
    db_path=DB_PATH,
    raw_data_dir=RAW_DATA_DIR,
    start_date=START_DATE,
    end_date=END_DATE,
    tickers=TICKERS,
    refresh_universe=REFRESH_UNIVERSE,
)
result

## 4. Check SQLite Outputs

Run lightweight sanity queries against the local database before promoting the same workflow to the CLI script.

In [ ]:
import sqlite3

connection = sqlite3.connect(DB_PATH)
connection.row_factory = sqlite3.Row

tables = connection.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
).fetchall()
[row["name"] for row in tables]

In [ ]:
for table_name in [
    "companies",
    "index_constituents",
    "price_bars",
    "fundamental_facts",
    "raw_artifacts",
]:
    count = connection.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"{table_name}: {count}")

In [ ]:
connection.execute(
    """
    SELECT c.ticker, c.name, c.cik, ic.weight, ic.as_of_date
    FROM companies AS c
    JOIN index_constituents AS ic ON c.company_id = ic.company_id
    ORDER BY ic.rank
    """
).fetchall()

## 5. Script Equivalent

Once the notebook flow looks right, the same path can be run from `scripts/ingest_data.py`.

In [ ]:
!python ../scripts/ingest_data.py --mode fixture --tickers MSFT NVDA